In [ ]:
#Phylogenetic tree visualization and phyloK matrix (ASTRAL species tree)
#author: Sheng-Kai Hsu
#last edited: 2025.04.15
#(sequence extraction + genetic-distance calc moved to
# src/S04_angiosperm353_extractAndDist.R, invoked from 05A_treeConstruction;
# supplemental cross-project comparisons moved to
# archived/05B_supplementalTreeComparisons.ipynb)
rm(list=ls())
PHYLOGWAS_ROOT <- Sys.getenv("PHYLOGWAS_ROOT", unset = "/workdir/sh2246/p_phyloGWAS")

library(ape)
library(ggplot2)
library(ggtree)

source(file.path(PHYLOGWAS_ROOT, "src/S01_phyloK.R"))

# ASTRAL tree visualization

In [ ]:
metadata = read.delim(file.path(PHYLOGWAS_ROOT, "data/Poaceae_metadata_highErrorFiltered_2025.10.08.tsv"),header = T)[,c(1,3)]
metadata$spTaxa = paste(metadata[,1],metadata[,2],sep = ":")

In [ ]:
# astral tree
spTre = read.tree(file.path(PHYLOGWAS_ROOT, "output/PoaceaeTree_angiosperm353.nwk"))
spTre.rooted = root(spTre,"ASM1935983v1")
spTre.rooted$edge.length[is.na(spTre.rooted$edge.length)] = 0.01

In [27]:
retainedSp = intersect(metadata$assemblyID,spTre.rooted$tip.label)

In [29]:
setdiff(metadata$assemblyID,spTre.rooted$tip.label)

character(0)

In [28]:
spTre.rooted = keep.tip(spTre.rooted,retainedSp)

In [ ]:
write.tree(spTre.rooted,file.path(PHYLOGWAS_ROOT, "output/PoaceaeTree_angiosperm353_astral_filtered.nwk"))

In [30]:
spTre.rooted.renamed=spTre.rooted
merge_tab5=data.frame(assemblyID=spTre.rooted$tip.label)
merge_tab5=merge(merge_tab5,metadata,by = "assemblyID")
merge_tab5=merge_tab5[!duplicated(merge_tab5$assemblyID),]
rownames(merge_tab5)=merge_tab5$assemblyID
merge_tab5=merge_tab5[spTre.rooted$tip.label,]

spTre.rooted.renamed$tip.label[!is.na(merge_tab5$spTaxa)] = merge_tab5$spTaxa[!is.na(merge_tab5$spTaxa)]

In [ ]:
write.tree(spTre.rooted.renamed,file.path(PHYLOGWAS_ROOT, "output/PoaceaeTree_angiosperm353_astral_spLabeled.nwk"))

In [31]:
spTre.rooted.renamed$node.label = round(as.numeric(spTre.rooted.renamed$node.label),2)

In [ ]:
png(file.path(PHYLOGWAS_ROOT, "output/angiosperm353_astral_spLabeled.png"),
    height = 60,width = 20, unit = "cm",pointsize = 6,res = 600)
ggtree(spTre.rooted.renamed,size =.3,ladderize = T ) + 
xlim(c(0,25)) +
geom_tiplab(hjust = -.025,size=1) +
geom_nodelab(hjust = 0,nudge_x = -0.05,color = "red",size = 1)+
theme(legend.position = c(.1,.95),
      legend.text = element_text(size = 10),
      legend.title = element_text(size = 12)) 
dev.off()

# phyloK matrix

In [ ]:
K = phyloK(spTre.rooted)

write.table(K,file.path(PHYLOGWAS_ROOT, "output/phyloK_728Poaceae_astral_20250407.txt"),sep = "\t",quote = F)